# QTradeSystematic - ML-Factor End-to-End Flow
End-to-end: data load -> feature engineering -> XGB model -> backtest -> CSV + pyfolio tearsheet export.


In [1]:
import sys
import warnings
from datetime import date, datetime
from pathlib import Path

repo_root = Path.cwd().resolve()
while not (repo_root / "qts").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

if not (repo_root / "qts").exists():
    raise FileNotFoundError("Could not find the repository root containing the qts package.")

repo_root_str = str(repo_root)
if repo_root_str not in sys.path:
    sys.path.insert(0, repo_root_str)

warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score, 
    log_loss
)
from typing import Any
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neural_network import MLPClassifier
import xgboost as xgb

from qts.config.builder import Config
from qts.orchestration.runtime import build_data_manager
from qts.research.backtest.base import BacktestConfig
from qts.research.backtest.pyfolio_adapter import (
    positions_frame,
    returns_series,
    transactions_frame,
)
from qts.research.backtest.tearsheet import save_tearsheet
from qts.research.features.fundamentals import prepare_vn_annual_fundamental_features
from qts.research.features.pipeline import FeaturePipeline
from qts.utils.export import export_portfolio_snapshots, export_trade_log
from qts.utils.labels import MLFactorClassThresholds, class_labels_from_probabilities
from qts.utils.paths import backtest_exports_dir, tearsheet_dir

print(f"Imports OK | repo_root={repo_root}")


Imports OK | repo_root=/Users/s2997726/Desktop/code/quant/QS/QTradeSystematic


## Step 1 - Load YAML config


In [2]:
CONFIG_PATH = repo_root / "configs" / "strategies" / "ml_factor" / "vn100_ml_example.yaml"
print(f"Using config: {CONFIG_PATH}")

resolved = Config.build(str(CONFIG_PATH))
config: BacktestConfig = resolved.raw

symbols = [
    *config.universe.stock,
    *config.universe.vn_stock,
    *config.universe.vn_warrant,
    *config.universe.vn_futures,
    *config.universe.crypto,
    *config.universe.crypto_futures,
]

print(f"Workflow       : {config.workflow}")
print(f"Universe       : {symbols}")
print(f"Date range     : {config.start_date} -> {config.end_date}")
print(f"Test start     : {config.test_start_date}")
print(f"Initial capital: {config.initial_capital:,}")
print(f"Engine         : {config.backtest_engine}")
print(f"Benchmark      : {config.benchmark}")


Using config: /Users/s2997726/Desktop/code/quant/QS/QTradeSystematic/configs/strategies/ml_factor/vn100_ml_example.yaml
Workflow       : research
Universe       : ['VN:ACB', 'VN:CTS', 'VN:DBC', 'VN:SHB', 'VN:VCG', 'VN:VIX', 'VN:VND', 'VN:DCM', 'VN:DGC', 'VN:ANV', 'VN:BID', 'VN:BMP', 'VN:BSI', 'VN:BVH', 'VN:CII', 'VN:CMG', 'VN:CTD', 'VN:CTG', 'VN:DIG', 'VN:DPM', 'VN:DXG', 'VN:EIB', 'VN:FPT', 'VN:GAS', 'VN:GMD', 'VN:HAG', 'VN:HCM', 'VN:HDC', 'VN:HDG', 'VN:HPG', 'VN:HSG', 'VN:HT1', 'VN:IMP', 'VN:KBC', 'VN:KDC', 'VN:KDH', 'VN:MBB', 'VN:MSN', 'VN:MWG', 'VN:NKG', 'VN:NLG', 'VN:NT2', 'VN:PAN', 'VN:PDR', 'VN:PHR', 'VN:PNJ', 'VN:PPC', 'VN:PTB', 'VN:PVD', 'VN:PVT', 'VN:REE', 'VN:SBT', 'VN:SJS', 'VN:SSI', 'VN:STB', 'VN:TLG', 'VN:VCB', 'VN:VHC', 'VN:VIC', 'VN:VNM', 'VN:VSC', 'VN:HHV', 'VN:TCH', 'VN:PC1', 'VN:SAB', 'VN:FTS', 'VN:VIB', 'VN:VJC', 'VN:PLX', 'VN:DGW', 'VN:SCS', 'VN:VCI', 'VN:BWE', 'VN:VPB', 'VN:LPB', 'VN:VPI', 'VN:VRE', 'VN:KOS', 'VN:CTR', 'VN:HDB', 'VN:GEX', 'VN:BCM', 'VN:POW', 'VN:FR

## Step 2 - Fetch Data from DuckDB


### Price data

In [3]:
data_manager = build_data_manager(resolved)

raw: pl.DataFrame = await data_manager.get_ohlcv(
    symbols=symbols,
    start=config.start_date,
    end=config.end_date,
)

print(f"Rows: {len(raw):,}  |  Symbols: {raw['symbol'].n_unique()}  |  Columns: {raw.columns}")
raw.tail(10)


Rows: 200,574  |  Symbols: 100  |  Columns: ['date', 'symbol', 'open', 'high', 'low', 'close', 'volume']


date,symbol,open,high,low,close,volume
date,str,f64,f64,f64,f64,f64
2026-05-29,"""VN:VIC""",211.0,214.3,209.5,211.3,2.017e6
2026-05-29,"""VN:VIX""",17.4,18.1,17.35,17.7,4.90198e7
2026-05-29,"""VN:VJC""",129.993,132.301,129.608,132.224,998700.0
2026-05-29,"""VN:VND""",17.45,17.5,16.85,16.85,1.47052e7
2026-05-29,"""VN:VNM""",59.0,59.3,58.9,59.2,1.9607e6
2026-05-29,"""VN:VPB""",27.2,27.4,27.05,27.1,1.25371e7
2026-05-29,"""VN:VPI""",62.2,62.5,62.0,62.3,1.8184e6
2026-05-29,"""VN:VRE""",32.25,32.45,31.7,32.25,3.6253e6
2026-05-29,"""VN:VSC""",20.15,20.4,20.0,20.0,3.0318e6


### Fundamental data

In [4]:
# from qts.research.features.fundamentals import prepare_vn_fundamental_features  

# fundamentals: pl.DataFrame = await data_manager.fetch_vn_fundamentals(
#     symbols=symbols,
#     start=config.start_date,
#     end=config.end_date,
#     show_progress=True,
# )


# annual_features = prepare_vn_fundamental_features(fundamentals, mode=1)
# quarterly_features = prepare_vn_fundamental_features(fundamentals, mode=2)


## Step 3 - Feature Engineering


In [5]:
feature_pipeline: FeaturePipeline = resolved.feature_pipeline
featured: pl.DataFrame = feature_pipeline.fit_transform(raw)

print(f"Featured shape : {featured.shape}")
print(f"Feature columns: {[col for col in featured.columns if col not in raw.columns]}")
featured

Featured shape : (200574, 28)
Feature columns: ['avg_price', 'log_volume', 'log_price', 'forward_return_12', 'forward_return_12_class', 'rsi_14', 'rsi_21', 'roc_14', 'roc_21', 'ma_5', 'ma_10', 'ma_20', 'adx_14', 'adx_21', 'vol_ratio_5', 'vol_ratio_12', 'hist_vol_21', 'hist_vol_42', 'zscore_10', 'zscore_21', 'zscore_42']


date,symbol,open,high,low,close,volume,avg_price,log_volume,log_price,forward_return_12,forward_return_12_class,rsi_14,rsi_21,roc_14,roc_21,ma_5,ma_10,ma_20,adx_14,adx_21,vol_ratio_5,vol_ratio_12,hist_vol_21,hist_vol_42,zscore_10,zscore_21,zscore_42
date,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,i16,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2018-01-02,"""VN:ACB""",6.49276,6.826172,6.457664,6.808624,3.657426e6,6.697487,15.11227,1.901732,0.018041,3,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2018-01-03,"""VN:ACB""",6.808624,6.861268,6.66824,6.791076,5.056543e6,6.773528,15.436194,1.913022,0.020672,3,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2018-01-04,"""VN:ACB""",6.791076,6.826172,6.738432,6.808624,6.365641e6,6.791076,15.666425,1.915609,0.036082,3,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2018-01-05,"""VN:ACB""",6.84372,6.93146,6.738432,6.808624,6.453452e6,6.826172,15.680126,1.920764,0.06701,3,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2018-01-08,"""VN:ACB""",6.808624,7.036748,6.808624,7.036748,3.879771e6,6.960707,15.171287,1.940281,0.024938,3,null,null,null,null,1.918282,null,null,null,null,0.984301,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2026-05-25,"""VN:VTP""",67.2,68.3,66.1,66.8,291500.0,67.066667,12.582795,4.205687,null,null,46.241531,46.095559,0.002261,-0.010851,4.234404,4.213258,4.210909,10.535483,7.876925,0.92332,0.942138,0.023572,0.024895,-0.387612,-0.285859,-0.780575
2026-05-26,"""VN:VTP""",67.4,67.4,66.2,66.5,315200.0,66.7,12.660963,4.200205,null,null,45.220008,45.426669,0.002156,-0.004921,4.220598,4.21739,4.209554,9.853326,7.548595,0.959561,0.948441,0.022707,0.024916,-0.723048,-0.409061,-0.972501
2026-05-27,"""VN:VTP""",66.6,68.8,66.6,66.8,453600.0,67.4,13.024971,4.210645,null,null,47.59426,46.9655,0.003203,-0.003945,4.213983,4.218384,4.2079,9.753005,7.467455,0.994834,0.979545,0.022786,0.024336,-0.532465,-0.216063,-0.888626


In [6]:
featured.columns

['date',
 'symbol',
 'open',
 'high',
 'low',
 'close',
 'volume',
 'avg_price',
 'log_volume',
 'log_price',
 'forward_return_12',
 'forward_return_12_class',
 'rsi_14',
 'rsi_21',
 'roc_14',
 'roc_21',
 'ma_5',
 'ma_10',
 'ma_20',
 'adx_14',
 'adx_21',
 'vol_ratio_5',
 'vol_ratio_12',
 'hist_vol_21',
 'hist_vol_42',
 'zscore_10',
 'zscore_21',
 'zscore_42']

## Step 5 - Dataset

In [14]:
featured_pd = featured.to_pandas().copy()
featured_pd["date"] = pd.to_datetime(featured_pd["date"])
featured_pd = featured_pd.sort_values(["date", "symbol"]).reset_index(drop=True)

FEATURE_COLS = [
    "log_volume",
    "log_price",
    "rsi_14",
    "rsi_21",
    "roc_14",
    "roc_21",
    "ma_5",
    "ma_10",
    "ma_20",
    "adx_14",
    "adx_21",
    "vol_ratio_5",
    "vol_ratio_12",
    "hist_vol_21",
    "hist_vol_42",
    "zscore_10",
    "zscore_21",
    "zscore_42",
]

LABEL_COL = next(
    col
    for col in featured_pd.columns
    if col.startswith("forward_return_") and col.endswith("_class")
)
TARGET_COL = LABEL_COL.removesuffix("_class")

missing_features = [col for col in FEATURE_COLS if col not in featured_pd.columns]
if missing_features:
    raise KeyError(f"Missing expected feature columns: {missing_features}")

MODEL_FEATURE_COLS = FEATURE_COLS.copy()
DATASET_COLS = ["date", "symbol", TARGET_COL, LABEL_COL, *MODEL_FEATURE_COLS]
dataset_pd = featured_pd[DATASET_COLS].copy()
dataset_pd = dataset_pd.dropna(subset=[TARGET_COL, LABEL_COL, *MODEL_FEATURE_COLS]).reset_index(drop=True)

print(f"Label column    : {LABEL_COL}")
print(f"Target column   : {TARGET_COL}")
print(f"Model features  : {len(MODEL_FEATURE_COLS)}")
print(f"Dataset rows    : {len(dataset_pd):,}")
print(MODEL_FEATURE_COLS)
print("Precomputed full-sample label distribution (reference only):")
print(dataset_pd[LABEL_COL].value_counts(dropna=False).sort_index())
dataset_pd[[TARGET_COL, LABEL_COL]].head(10)


Label column    : forward_return_12_class
Target column   : forward_return_12
Model features  : 18
Dataset rows    : 195,174
['log_volume', 'log_price', 'rsi_14', 'rsi_21', 'roc_14', 'roc_21', 'ma_5', 'ma_10', 'ma_20', 'adx_14', 'adx_21', 'vol_ratio_5', 'vol_ratio_12', 'hist_vol_21', 'hist_vol_42', 'zscore_10', 'zscore_21', 'zscore_42']
Precomputed full-sample label distribution (reference only):
forward_return_12_class
0.0    29208
1.0    58666
2.0    19634
3.0    58778
4.0    28888
Name: count, dtype: int64


,forward_return_12,forward_return_12_class
0,0.148565,4.0
1,-0.028112,1.0
2,-0.053719,1.0
3,-0.039572,1.0
4,0.082927,3.0
5,0.061156,3.0
6,0.090149,4.0
7,-0.020576,1.0
8,0.086957,4.0
9,0.018077,3.0


## Step 6 - Models


### XGB Classifier

In [15]:
class XGBPredictor:
    def __init__(
        self,
        *,
        max_depth: int = 3,
        n_estimators: int = 200,
        learning_rate: float = 0.05,
        subsample: float = 0.9,
        colsample_bytree: float = 0.9,
        reg_alpha: float = 0.0,
        reg_lambda: float = 1.0,
        eval_metric: str = "mlogloss",
        seed: int = 42,
        early_stopping_rounds: int = 20,
    ) -> None:
        self.params = {
            "objective": "multi:softprob",
            "num_class": 5,
            "tree_method": "hist",
            "max_depth": max_depth,
            "n_estimators": n_estimators,
            "learning_rate": learning_rate,
            "subsample": subsample,
            "colsample_bytree": colsample_bytree,
            "reg_alpha": reg_alpha,
            "reg_lambda": reg_lambda,
            "eval_metric": eval_metric,
            "random_state": seed,
            "n_jobs": -1,
            "early_stopping_rounds": early_stopping_rounds,
        }
        self.model: xgb.XGBClassifier | None = None
        self.constant_class: int | None = None

    def fit(
        self,
        X: pd.DataFrame,
        y: pd.Series,
        X_valid: pd.DataFrame | None = None,
        y_valid: pd.Series | None = None,
    ) -> None:
        y_int = pd.Series(y, copy=False).astype(int)
        unique = np.sort(y_int.unique())

        if len(unique) == 1:
            self.constant_class = int(unique[0])
            self.model = None
            return

        self.constant_class = None
        self.model = xgb.XGBClassifier(**self.params)

        if X_valid is not None and y_valid is not None and len(X_valid) > 0:
            self.model.fit(
                X,
                y_int,
                eval_set=[(X_valid, pd.Series(y_valid, copy=False).astype(int))],
                verbose=False,
            )
        else:
            self.model.fit(X, y_int, verbose=False)

    def predict_proba(self, X: pd.DataFrame) -> pd.DataFrame:
        if self.constant_class is not None:
            probs = np.zeros((len(X), 5), dtype=float)
            probs[:, self.constant_class] = 1.0
            return pd.DataFrame(probs, index=X.index, columns=[f"prob_{i}" for i in range(5)])

        if self.model is None:
            raise RuntimeError("XGBPredictor.fit must be called before predict_proba.")

        raw_probs = np.asarray(self.model.predict_proba(X), dtype=float)
        out = np.zeros((len(X), 5), dtype=float)
        for idx, cls in enumerate(self.model.classes_):
            out[:, int(cls)] = raw_probs[:, idx]
        return pd.DataFrame(out, index=X.index, columns=[f"prob_{i}" for i in range(5)])

### ANN Classifier

In [16]:

class ANNPredictor:
    def __init__(
        self,
        *,
        hidden_layer_sizes: tuple[int, ...] = (128, 64, 32),
        activation: str = "relu",
        alpha: float = 1e-4,
        learning_rate_init: float = 1e-3,
        max_iter: int = 300,
        random_state: int = 42,
    ) -> None:
        self.pipeline = Pipeline(
            steps=[
                ("impute", SimpleImputer(strategy="median")),
                ("scale", StandardScaler()),
                (
                    "mlp",
                    MLPClassifier(
                        hidden_layer_sizes=hidden_layer_sizes,
                        activation=activation,
                        solver="adam",
                        alpha=alpha,
                        learning_rate_init=learning_rate_init,
                        max_iter=max_iter,
                        early_stopping=True,
                        validation_fraction=0.1,
                        n_iter_no_change=10,
                        random_state=random_state,
                    ),
                ),
            ]
        )
        self.constant_class: int | None = None

    def fit(
        self,
        X: pd.DataFrame,
        y: pd.Series,
        X_valid: pd.DataFrame | None = None,
        y_valid: pd.Series | None = None,
    ) -> None:
        y_int = pd.Series(y, copy=False).astype(int)
        unique = np.sort(y_int.unique())

        if len(unique) == 1:
            self.constant_class = int(unique[0])
            return

        self.constant_class = None
        self.pipeline.fit(X, y_int)

    def predict_proba(self, X: pd.DataFrame) -> pd.DataFrame:
        if self.constant_class is not None:
            probs = np.zeros((len(X), 5), dtype=float)
            probs[:, self.constant_class] = 1.0
            return pd.DataFrame(probs, index=X.index, columns=[f"prob_{i}" for i in range(5)])

        raw_probs = np.asarray(self.pipeline.predict_proba(X), dtype=float)
        classes = self.pipeline.named_steps["mlp"].classes_

        out = np.zeros((len(X), 5), dtype=float)
        for idx, cls in enumerate(classes):
            out[:, int(cls)] = raw_probs[:, idx]
        return pd.DataFrame(out, index=X.index, columns=[f"prob_{i}" for i in range(5)])

## Step 7 - Walk-forward training 

In [21]:
import importlib
import qts as qts_lib

qts_labels = importlib.reload(qts_lib)
MLFactorClassThresholds = qts_lib.utils.labels.MLFactorClassThresholds
class_labels_from_probabilities = qts_lib.utils.labels.class_labels_from_probabilities

CLASS_SCORES = np.array([-2.0, -1.0, 0.0, 1.0, 2.0], dtype=float)

def probability_weighted_score(probabilities: np.ndarray) -> np.ndarray:
    return probabilities @ CLASS_SCORES

def build_rebalance_dates(frame: pd.DataFrame, rebalance_period: int) -> list[pd.Timestamp]:
    dates = pd.DatetimeIndex(pd.to_datetime(frame["date"]).drop_duplicates()).sort_values()
    return list(dates[::rebalance_period])

def split_train_valid_by_date(train_frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    unique_dates = np.array(sorted(pd.to_datetime(train_frame["date"]).drop_duplicates()))
    if len(unique_dates) < 3:
        return train_frame.copy(), train_frame.iloc[0:0].copy()

    n_splits = min(5, len(unique_dates) - 1)
    splitter = TimeSeriesSplit(n_splits=n_splits)
    train_idx, valid_idx = list(splitter.split(unique_dates))[-1]

    train_dates = set(unique_dates[train_idx])
    valid_dates = set(unique_dates[valid_idx])

    train_part = train_frame.loc[train_frame["date"].isin(train_dates)].copy()
    valid_part = train_frame.loc[train_frame["date"].isin(valid_dates)].copy()
    return train_part, valid_part

def apply_feature_coverage(frame: pd.DataFrame, predictor_cols: list[str]) -> pd.DataFrame:
    min_non_null = max(3, len(predictor_cols) // 3)
    coverage = frame[predictor_cols].notna().sum(axis=1) >= min_non_null
    return frame.loc[coverage].copy()

def apply_window_labels(
    frame: pd.DataFrame,
    *,
    target_col: str,
    label_col: str,
    thresholds: MLFactorClassThresholds,
) -> pd.DataFrame:
    labeled = frame.copy()
    labeled[label_col] = thresholds.transform(labeled[target_col])
    return labeled.loc[labeled[label_col] >= 0].copy()

def walk_forward_predict(
    frame: pd.DataFrame,
    predictor: Any,
    predictor_cols: list[str],
    label_col: str,
    target_col: str,
    train_window: int,
    rebalance_period: int,
    prediction_start_date: pd.Timestamp | None = None,
    prediction_end_date: pd.Timestamp | None = None,
) -> pd.DataFrame:
    data = frame.sort_values(["date", "symbol"]).copy()
    all_dates = list(pd.DatetimeIndex(pd.to_datetime(data["date"]).drop_duplicates()).sort_values())
    rebalance_dates = build_rebalance_dates(data, rebalance_period)

    if prediction_start_date is not None:
        rebalance_dates = [d for d in rebalance_dates if d >= prediction_start_date]
    if prediction_end_date is not None:
        rebalance_dates = [d for d in rebalance_dates if d <= prediction_end_date]

    rows: list[pd.DataFrame] = []

    for rebalance_date in rebalance_dates:
        idx = all_dates.index(rebalance_date)
        latest_trainable_idx = idx - rebalance_period
        if latest_trainable_idx < 0:
            continue

        trainable_dates = all_dates[: latest_trainable_idx + 1]
        if len(trainable_dates) < train_window:
            continue

        train_dates = set(trainable_dates[-train_window:])

        train_slice = data.loc[data["date"].isin(train_dates)].copy()
        train_slice = train_slice.loc[train_slice[target_col].notna()].copy()
        train_slice = apply_feature_coverage(train_slice, predictor_cols)

        predict_slice = data.loc[data["date"] == rebalance_date].copy()
        predict_slice = predict_slice.loc[predict_slice[target_col].notna()].copy()
        predict_slice = apply_feature_coverage(predict_slice, predictor_cols)

        if train_slice.empty or predict_slice.empty:
            continue

        train_part_raw, valid_part_raw = split_train_valid_by_date(train_slice)
        threshold_source = train_part_raw if not train_part_raw.empty else train_slice
        thresholds = MLFactorClassThresholds.fit(threshold_source[target_col], target_col=target_col)

        train_part = apply_window_labels(
            train_part_raw,
            target_col=target_col,
            label_col=label_col,
            thresholds=thresholds,
        )
        valid_part = apply_window_labels(
            valid_part_raw,
            target_col=target_col,
            label_col=label_col,
            thresholds=thresholds,
        )
        predict_slice = apply_window_labels(
            predict_slice,
            target_col=target_col,
            label_col=label_col,
            thresholds=thresholds,
        )

        if train_part.empty or predict_slice.empty:
            continue

        X_train = train_part[predictor_cols]
        y_train = train_part[label_col].astype(int)

        X_valid = None
        y_valid = None
        if not valid_part.empty and valid_part[label_col].notna().any():
            X_valid = valid_part[predictor_cols]
            y_valid = valid_part[label_col].astype(int)

        predictor.fit(X_train, y_train, X_valid=X_valid, y_valid=y_valid)

        prob_frame = predictor.predict_proba(predict_slice[predictor_cols])
        prob_values = prob_frame.to_numpy(dtype=float)
        scores = probability_weighted_score(prob_values)
        pred_class = class_labels_from_probabilities(prob_values)

        out = pd.DataFrame(
            {
                "date": predict_slice["date"].to_numpy(),
                "symbol": predict_slice["symbol"].to_numpy(),
                "pred_class": pred_class.astype(int),
                "score": scores.astype(float),
                "realized_forward_return": predict_slice[target_col].to_numpy(),
                "true_label": predict_slice[label_col].to_numpy(),
            },
            index=predict_slice.index,
        )
        out = pd.concat([out, prob_frame], axis=1)
        rows.append(out.reset_index(drop=True))

    if not rows:
        return pd.DataFrame(
            columns=[
                "date",
                "symbol",
                "pred_class",
                "score",
                "realized_forward_return",
                "true_label",
                "prob_0",
                "prob_1",
                "prob_2",
                "prob_3",
                "prob_4",
            ]
        )

    return pd.concat(rows, ignore_index=True).sort_values(
        ["date", "score"], ascending=[True, False]
    ).reset_index(drop=True)

In [22]:
def evaluate_predictions(predictions: pd.DataFrame) -> tuple[dict[str, float], pd.Series]:
    scored = predictions.loc[predictions["true_label"].notna()].copy()
    if scored.empty:
        return {
            "rows": 0.0,
            "log_loss": np.nan,
            "accuracy": np.nan,
            "balanced_accuracy": np.nan,
        }, pd.Series(dtype=float)

    y_true = scored["true_label"].astype(int)
    y_pred = scored["pred_class"].astype(int)
    y_prob = scored[[f"prob_{i}" for i in range(5)]].to_numpy(dtype=float)

    metrics = {
        "rows": float(len(scored)),
        "log_loss": float(log_loss(y_true, y_prob, labels=[0, 1, 2, 3, 4])),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }

    mean_returns = scored.groupby("pred_class")["realized_forward_return"].mean().sort_index()
    return metrics, mean_returns


## Step 8 -  Run models

In [23]:
TRAIN_WINDOW = int(config.train_window)
REBALANCE_PERIOD = int(config.rebalance_frequency)
TEST_START = pd.Timestamp(config.test_start_date) if config.test_start_date else None

print(f"Train window     : {TRAIN_WINDOW} days")
print(f"Rebalance period : {REBALANCE_PERIOD} days")
print(f"Test start date  : {TEST_START}")

Train window     : 252 days
Rebalance period : 12 days
Test start date  : 2023-01-01 00:00:00


### XGB

In [24]:
xgb_predictions = walk_forward_predict(
    frame=dataset_pd,
    predictor=XGBPredictor(),
    predictor_cols=MODEL_FEATURE_COLS,
    label_col=LABEL_COL,
    target_col=TARGET_COL,
    train_window=TRAIN_WINDOW,
    rebalance_period=REBALANCE_PERIOD,
    prediction_start_date=TEST_START if TEST_START is not None else None,
    prediction_end_date=None,
)

print(f"Prediction rows : {len(xgb_predictions):,}")
print(f"Signal dates    : {xgb_predictions['date'].nunique()}")
print("Class distribution:")
print(xgb_predictions["pred_class"].value_counts().sort_index())
print("Window-relative true-label distribution:")
print(xgb_predictions["true_label"].value_counts().sort_index())

xgb_predictions.head(10)

Prediction rows : 6,961
Signal dates    : 70
Class distribution:
pred_class
1      27
2    6884
3      50
Name: count, dtype: int64
Window-relative true-label distribution:
true_label
0     987
1    2105
2     750
3    2136
4     983
Name: count, dtype: int64


,date,symbol,pred_class,score,realized_forward_return,true_label,prob_0,prob_1,prob_2,prob_3,prob_4
0,2023-01-06,VN:FTS,3,0.867271,0.099980,4,0.094566,0.155211,0.042574,0.203683,0.503966
1,2023-01-06,VN:DXS,3,0.809595,0.078125,3,0.130423,0.119234,0.050567,0.209879,0.489898
2,2023-01-06,VN:VIX,3,0.752852,0.123788,4,0.107983,0.152279,0.051086,0.256208,0.432445
3,2023-01-06,VN:NKG,3,0.604134,0.200000,4,0.116776,0.189546,0.052448,0.255230,0.386000
4,2023-01-06,VN:LPB,3,0.603768,0.024286,3,0.105902,0.192519,0.068139,0.258788,0.374652
5,2023-01-06,VN:SJS,3,0.600650,0.089671,4,0.079146,0.162890,0.125296,0.343506,0.289162
6,2023-01-06,VN:DXG,3,0.594762,0.128788,4,0.125212,0.186923,0.051249,0.241125,0.395492
7,2023-01-06,VN:PAN,3,0.579842,0.084112,4,0.102263,0.197333,0.062615,0.293877,0.343912
8,2023-01-06,VN:SIP,3,0.574882,0.296422,4,0.085957,0.190636,0.107211,0.294963,0.321234
9,2023-01-06,VN:CTS,3,0.574586,0.105839,4,0.117662,0.203798,0.052999,0.237371,0.388169


In [25]:
xgb_predictions.tail(200)

,date,symbol,pred_class,score,realized_forward_return,true_label,prob_0,prob_1,prob_2,prob_3,prob_4
6761,2026-04-21,VN:DXS,3,0.728702,0.154930,4,0.091141,0.106726,0.087691,0.411175,0.303267
6762,2026-04-21,VN:MSB,3,0.533734,0.076305,3,0.049877,0.270522,0.069520,0.316153,0.293928
6763,2026-04-21,VN:HSG,2,0.351808,-0.005257,1,0.084804,0.292283,0.061389,0.309348,0.252176
6764,2026-04-21,VN:VPI,2,0.255363,0.018062,3,0.087386,0.286860,0.072733,0.389049,0.163973
6765,2026-04-21,VN:ACB,2,0.224324,-0.044240,1,0.065580,0.304615,0.114884,0.369743,0.145178
...,...,...,...,...,...,...,...,...,...,...,...
6956,2026-05-12,VN:EVF,2,-0.330560,0.026316,3,0.199151,0.366416,0.090695,0.253318,0.090420
6957,2026-05-12,VN:VSC,2,-0.360627,-0.083900,0,0.233017,0.329391,0.094366,0.251652,0.091573
6958,2026-05-12,VN:DGW,2,-0.387143,0.008485,2,0.175130,0.420637,0.101867,0.220975,0.081390
6959,2026-05-12,VN:VIC,2,-0.455604,-0.050450,1,0.356656,0.258694,0.052175,0.148547,0.183928


In [26]:
xgb_eval, xgb_class_returns = evaluate_predictions(xgb_predictions)

print("=== XGB prediction metrics ===")
for key, value in xgb_eval.items():
    print(f"  {key:18s}: {value:.6f}" if pd.notna(value) else f"  {key:18s}: nan")

print("\n=== Mean realized forward return by predicted class ===")
print(xgb_class_returns)

=== XGB prediction metrics ===
  rows              : 6961.000000
  log_loss          : 1.499177
  accuracy          : 0.109467
  balanced_accuracy : 0.200436

=== Mean realized forward return by predicted class ===
pred_class
1    0.024265
2    0.009747
3    0.033475
Name: realized_forward_return, dtype: float64


### ANN

In [27]:
ann_predictions = walk_forward_predict(
    frame=dataset_pd,
    predictor=ANNPredictor(),
    predictor_cols=MODEL_FEATURE_COLS,
    label_col=LABEL_COL,
    target_col=TARGET_COL,
    train_window=TRAIN_WINDOW,
    rebalance_period=REBALANCE_PERIOD,
    prediction_start_date=TEST_START if TEST_START is not None else None,
    prediction_end_date=None,
)

ann_eval, ann_class_returns = evaluate_predictions(ann_predictions)

print("=== ANN prediction metrics ===")
for key, value in ann_eval.items():
    print(f"  {key:18s}: {value:.6f}" if pd.notna(value) else f"  {key:18s}: nan")

print("\n=== ANN mean realized forward return by predicted class ===")
print(ann_class_returns)

=== ANN prediction metrics ===
  rows              : 6961.000000
  log_loss          : 2.720968
  accuracy          : 0.228846
  balanced_accuracy : 0.212435

=== ANN mean realized forward return by predicted class ===
pred_class
0    0.015924
1    0.011467
2    0.009515
3    0.005865
4    0.015117
Name: realized_forward_return, dtype: float64


In [28]:
ann_eval, ann_class_returns = evaluate_predictions(ann_predictions)

print("=== ANN prediction metrics ===")
for key, value in ann_eval.items():
    print(f"  {key:18s}: {value:.6f}" if pd.notna(value) else f"  {key:18s}: nan")

print("\n=== ANN mean realized forward return by predicted class ===")
print(ann_class_returns)

=== ANN prediction metrics ===
  rows              : 6961.000000
  log_loss          : 2.720968
  accuracy          : 0.228846
  balanced_accuracy : 0.212435

=== ANN mean realized forward return by predicted class ===
pred_class
0    0.015924
1    0.011467
2    0.009515
3    0.005865
4    0.015117
Name: realized_forward_return, dtype: float64


## Step 8 - Run backtest


In [29]:
from qts.research.strategies.base import BaseStrategy

ENTRY_PROB_4_MIN = 0.30
ENTRY_PROB_34_MIN = 0.60
MAX_POSITIONS = 3


class XGBTopKEntryStrategy(BaseStrategy):
    def __init__(
        self,
        predictions: pd.DataFrame,
        sessions: list[date],
        *,
        prob_4_min: float = ENTRY_PROB_4_MIN,
        prob_34_min: float = ENTRY_PROB_34_MIN,
        max_positions: int = MAX_POSITIONS,
    ) -> None:
        self.predictions = predictions.copy()
        self.sessions = sorted(pd.to_datetime(pd.Index(sessions)).date)
        self.prob_4_min = float(prob_4_min)
        self.prob_34_min = float(prob_34_min)
        self.max_positions = int(max_positions)

    def generate_signals(self, data: pl.DataFrame) -> pl.DataFrame:
        if self.predictions.empty:
            return self.empty_signal_frame()

        frame = self.predictions.copy()
        frame["date"] = pd.to_datetime(frame["date"]).dt.date
        frame["prob_34"] = frame["prob_3"] + frame["prob_4"]

        next_session_map = {
            current: next_session
            for current, next_session in zip(self.sessions[:-1], self.sessions[1:], strict=False)
        }

        rows: list[dict[str, object]] = []
        previous_symbols: set[str] = set()

        for prediction_date in sorted(frame["date"].drop_duplicates()):
            execution_date = next_session_map.get(prediction_date)
            if execution_date is None:
                continue

            daily = frame.loc[frame["date"] == prediction_date].copy()
            selected = (
                daily.loc[
                    (daily["prob_4"] > self.prob_4_min)
                    & ((daily["prob_3"] + daily["prob_4"]) > self.prob_34_min)
                ]
                .sort_values(["prob_4", "score"], ascending=False)
                .head(self.max_positions)
            )

            current_symbols = set(selected["symbol"].tolist())
            if not selected.empty:
                weight = 1.0 / float(self.max_positions)
                for record in selected.itertuples(index=False):
                    rows.append(
                        {
                            "date": execution_date,
                            "symbol": record.symbol,
                            "signal": 1,
                            "weight": weight,
                        }
                    )

            for symbol in sorted(previous_symbols - current_symbols):
                rows.append(
                    {
                        "date": execution_date,
                        "symbol": symbol,
                        "signal": 0,
                        "weight": 0.0,
                    }
                )

            previous_symbols = current_symbols

        if not rows:
            return self.empty_signal_frame()

        signals = pl.from_pandas(pd.DataFrame(rows))
        return self.validate_signal_frame(signals.sort(["date", "symbol"]))


xgb_signal_strategy = XGBTopKEntryStrategy(
    predictions=xgb_predictions,
    sessions=raw["date"].unique().to_list(),
)
xgb_entry_signals = xgb_signal_strategy.generate_signals(raw)

print(f"Signal rows     : {xgb_entry_signals.height:,}")
print(f"Signal dates    : {xgb_entry_signals['date'].n_unique()}")
print(xgb_entry_signals.tail(15))

engine = resolved.engine
print(f"Engine: {engine.__class__.__name__}")

result = engine.run(
    strategy=xgb_signal_strategy,
    data=raw,
    config=config,
)

print("\n=== Backtest metrics ===")
for key, value in result.metrics.items():
    print(f"  {key:20s}: {value:.4f}")


Signal rows     : 39
Signal dates    : 18
shape: (15, 4)
┌────────────┬────────┬────────┬──────────┐
│ date       ┆ symbol ┆ signal ┆ weight   │
│ ---        ┆ ---    ┆ ---    ┆ ---      │
│ date       ┆ str    ┆ i32    ┆ f64      │
╞════════════╪════════╪════════╪══════════╡
│ 2025-05-08 ┆ VN:FTS ┆ 1      ┆ 0.333333 │
│ 2025-05-08 ┆ VN:PHR ┆ 1      ┆ 0.333333 │
│ 2025-05-26 ┆ VN:BSI ┆ 0      ┆ 0.0      │
│ 2025-05-26 ┆ VN:FTS ┆ 0      ┆ 0.0      │
│ 2025-05-26 ┆ VN:GEE ┆ 1      ┆ 0.333333 │
│ …          ┆ …      ┆ …      ┆ …        │
│ 2026-04-06 ┆ VN:DXS ┆ 1      ┆ 0.333333 │
│ 2026-04-22 ┆ VN:DGC ┆ 0      ┆ 0.0      │
│ 2026-04-22 ┆ VN:DXS ┆ 1      ┆ 0.333333 │
│ 2026-05-13 ┆ VN:DXS ┆ 1      ┆ 0.333333 │
│ 2026-05-13 ┆ VN:MSB ┆ 1      ┆ 0.333333 │
└────────────┴────────┴────────┴──────────┘
Engine: VectorBTProEngine

=== Backtest metrics ===
  sharpe              : 0.1941
  sortino             : 0.0728
  cagr                : 0.0100
  max_drawdown        : 0.1306
  win_rate         

## Step 7 - Signal Win/Loss Audit


In [30]:
def _normalize_timestamp_series(series: pd.Series) -> pd.Series:
    timestamps = pd.to_datetime(series)
    if getattr(timestamps.dt, "tz", None) is not None:
        timestamps = timestamps.dt.tz_convert(None)
    return timestamps.dt.normalize()


signals_audit_pd = xgb_entry_signals.to_pandas().copy()
signals_audit_pd["date"] = pd.to_datetime(signals_audit_pd["date"]).dt.normalize()
signals_audit_pd = signals_audit_pd.sort_values(["symbol", "date"]).reset_index(drop=True)
signals_audit_pd["target"] = signals_audit_pd["signal"].astype(float) * signals_audit_pd["weight"].astype(float)
signals_audit_pd["prev_target"] = signals_audit_pd.groupby("symbol")["target"].shift(fill_value=0.0)

signals_audit_pd["event_type"] = np.select(
    [
        (signals_audit_pd["target"] > 0) & (signals_audit_pd["prev_target"] <= 0),
        (signals_audit_pd["target"] <= 0) & (signals_audit_pd["prev_target"] > 0),
        (signals_audit_pd["target"] > 0) & (~np.isclose(signals_audit_pd["target"], signals_audit_pd["prev_target"])),
    ],
    ["entry", "exit", "rebalance"],
    default="hold",
)

entry_signals_pd = (
    signals_audit_pd.loc[signals_audit_pd["event_type"] == "entry", ["date", "symbol", "weight"]]
    .rename(columns={"date": "entry_date", "weight": "signal_weight"})
    .reset_index(drop=True)
)

prediction_details_pd = xgb_predictions.copy()
prediction_details_pd["prediction_date"] = pd.to_datetime(prediction_details_pd["date"]).dt.normalize()
session_dates = pd.DatetimeIndex(pd.to_datetime(raw["date"].unique().to_list())).normalize().sort_values()
prediction_to_entry_pd = pd.DataFrame(
    {
        "prediction_date": session_dates[:-1],
        "entry_date": session_dates[1:],
    }
)
prediction_details_pd = prediction_details_pd.merge(prediction_to_entry_pd, on="prediction_date", how="left")
prediction_details_pd = prediction_details_pd[
    [
        "entry_date",
        "symbol",
        "pred_class",
        "score",
        "prob_0",
        "prob_1",
        "prob_2",
        "prob_3",
        "prob_4",
    ]
]

trade_log_pd = result.trade_log.to_pandas().copy()
if trade_log_pd.empty:
    signal_trade_outcomes_pd = entry_signals_pd.copy()
    signal_trade_outcomes_pd["exit_date"] = pd.NaT
    signal_trade_outcomes_pd["start_price"] = np.nan
    signal_trade_outcomes_pd["end_price"] = np.nan
    signal_trade_outcomes_pd["profit_pct"] = np.nan
    signal_trade_outcomes_pd["signal_result"] = "open"
else:
    trade_log_pd = trade_log_pd.rename(columns={"ticker": "symbol"})
    trade_log_pd["entry_date"] = _normalize_timestamp_series(trade_log_pd["entry_time"])
    trade_log_pd["exit_date"] = _normalize_timestamp_series(trade_log_pd["exit_time"])
    trade_log_pd["signal_result"] = np.select(
        [trade_log_pd["profit_pct"] > 0, trade_log_pd["profit_pct"] < 0],
        ["win", "loss"],
        default="flat",
    )
    trade_log_pd["holding_days"] = (trade_log_pd["exit_date"] - trade_log_pd["entry_date"]).dt.days

    signal_trade_outcomes_pd = entry_signals_pd.merge(
        trade_log_pd[
            [
                "entry_date",
                "exit_date",
                "symbol",
                "start_price",
                "end_price",
                "profit_pct",
                "holding_days",
                "signal_result",
            ]
        ],
        on=["entry_date", "symbol"],
        how="left",
    )

signal_trade_outcomes_pd = signal_trade_outcomes_pd.merge(
    prediction_details_pd,
    on=["entry_date", "symbol"],
    how="left",
)
signal_trade_outcomes_pd["signal_result"] = signal_trade_outcomes_pd["signal_result"].fillna("open")
signal_trade_outcomes_pd["profit_pct"] = signal_trade_outcomes_pd["profit_pct"].astype(float)
signal_trade_outcomes_pd["profit_bps"] = signal_trade_outcomes_pd["profit_pct"] * 10000.0
signal_trade_outcomes_pd = signal_trade_outcomes_pd.sort_values(["entry_date", "symbol"]).reset_index(drop=True)

signal_result_lookup = signal_trade_outcomes_pd[["entry_date", "symbol", "signal_result", "profit_pct", "exit_date"]].rename(
    columns={"entry_date": "date"}
)
signals_audit_pd = signals_audit_pd.merge(signal_result_lookup, on=["date", "symbol"], how="left")

closed_trade_mask = signal_trade_outcomes_pd["signal_result"].isin(["win", "loss", "flat"])
closed_trade_win_rate = (
    (signal_trade_outcomes_pd.loc[closed_trade_mask, "signal_result"] == "win").mean()
    if closed_trade_mask.any()
    else np.nan
)

print("Signal event counts:")
print(signals_audit_pd["event_type"].value_counts().sort_index())
print("\nRealized entry results:")
print(signal_trade_outcomes_pd["signal_result"].value_counts().sort_index())
print(f"\nClosed-trade win rate: {closed_trade_win_rate:.4f}" if pd.notna(closed_trade_win_rate) else "\nClosed-trade win rate: nan")

with pd.option_context("display.max_rows", None, "display.max_columns", None):
    print("\n=== Signal Table Audit ===")
    print(signals_audit_pd.to_string(index=False))
    print("\n=== Entry Signal Outcomes ===")
    print(signal_trade_outcomes_pd.to_string(index=False))


Signal event counts:
event_type
entry    18
exit     16
hold      5
Name: count, dtype: int64

Realized entry results:
signal_result
open    18
Name: count, dtype: int64

Closed-trade win rate: nan

=== Signal Table Audit ===
      date symbol  signal   weight   target  prev_target event_type signal_result  profit_pct exit_date
2025-01-15 VN:ANV       1 0.333333 0.333333     0.000000      entry          open         NaN       NaT
2025-02-07 VN:ANV       0 0.000000 0.000000     0.333333       exit           NaN         NaN       NaT
2025-05-08 VN:BSI       1 0.333333 0.333333     0.000000      entry          open         NaN       NaT
2025-05-26 VN:BSI       0 0.000000 0.000000     0.333333       exit           NaN         NaN       NaT
2025-02-25 VN:CTD       1 0.333333 0.333333     0.000000      entry          open         NaN       NaT
2025-03-13 VN:CTD       0 0.000000 0.000000     0.333333       exit           NaN         NaN       NaT
2026-04-06 VN:DGC       1 0.333333 0.333333   

## Step 8 - Save CSV outputs


In [31]:
run_id = f"xgb_topk_{result.engine_name}_{datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')}"
exports_dir = backtest_exports_dir()

trade_log_path = exports_dir / f"{run_id}_trade_log.csv"
snapshots_path = exports_dir / f"{run_id}_snapshots.csv"
signals_path = exports_dir / f"{run_id}_signals.csv"
signal_audit_path = exports_dir / f"{run_id}_signal_audit.csv"
signal_outcomes_path = exports_dir / f"{run_id}_signal_outcomes.csv"

export_trade_log(result, trade_log_path)
export_portfolio_snapshots(result, snapshots_path)
xgb_entry_signals.write_csv(signals_path)
signals_audit_pd.to_csv(signal_audit_path, index=False)
signal_trade_outcomes_pd.to_csv(signal_outcomes_path, index=False)

print(f"Trade log        -> {trade_log_path}  ({result.trade_log.height} rows)")
print(f"Snapshots        -> {snapshots_path}  ({result.portfolio_snapshots.height} rows)")
print(f"Signals          -> {signals_path}  ({xgb_entry_signals.height} rows)")
print(f"Signal audit     -> {signal_audit_path}  ({len(signals_audit_pd)} rows)")
print(f"Signal outcomes  -> {signal_outcomes_path}  ({len(signal_trade_outcomes_pd)} rows)")

signal_trade_outcomes_pd.head(10)


Trade log        -> /Users/s2997726/.qts/exports/backtest/xgb_topk_vectorbt_20260620T233517Z_trade_log.csv  (0 rows)
Snapshots        -> /Users/s2997726/.qts/exports/backtest/xgb_topk_vectorbt_20260620T233517Z_snapshots.csv  (0 rows)
Signals          -> /Users/s2997726/.qts/exports/backtest/xgb_topk_vectorbt_20260620T233517Z_signals.csv  (39 rows)
Signal audit     -> /Users/s2997726/.qts/exports/backtest/xgb_topk_vectorbt_20260620T233517Z_signal_audit.csv  (39 rows)
Signal outcomes  -> /Users/s2997726/.qts/exports/backtest/xgb_topk_vectorbt_20260620T233517Z_signal_outcomes.csv  (18 rows)


,entry_date,symbol,signal_weight,exit_date,start_price,end_price,profit_pct,signal_result,pred_class,score,prob_0,prob_1,prob_2,prob_3,prob_4,profit_bps
0,2023-01-09,VN:DXS,0.333333,NaT,NaN,NaN,NaN,open,3,0.809595,0.130423,0.119234,0.050567,0.209879,0.489898,NaN
1,2023-01-09,VN:FTS,0.333333,NaT,NaN,NaN,NaN,open,3,0.867271,0.094566,0.155211,0.042574,0.203683,0.503966,NaN
2,2023-01-09,VN:VIX,0.333333,NaT,NaN,NaN,NaN,open,3,0.752852,0.107983,0.152279,0.051086,0.256208,0.432445,NaN
3,2023-02-17,VN:PDR,0.333333,NaT,NaN,NaN,NaN,open,3,0.662702,0.169704,0.123947,0.048375,0.189891,0.468083,NaN
4,2024-10-22,VN:IMP,0.333333,NaT,NaN,NaN,NaN,open,3,0.581829,0.109719,0.199987,0.044734,0.289864,0.355695,NaN
5,2024-11-25,VN:HT1,0.333333,NaT,NaN,NaN,NaN,open,3,0.809145,0.048893,0.184379,0.089876,0.262392,0.414459,NaN
6,2025-01-15,VN:ANV,0.333333,NaT,NaN,NaN,NaN,open,3,0.799939,0.083174,0.140072,0.069728,0.307693,0.399333,NaN
7,2025-01-15,VN:HDB,0.333333,NaT,NaN,NaN,NaN,open,3,0.782842,0.088026,0.144838,0.055751,0.319037,0.392348,NaN
8,2025-01-15,VN:PDR,0.333333,NaT,NaN,NaN,NaN,open,3,0.833439,0.076136,0.135022,0.083857,0.289237,0.415748,NaN
9,2025-02-25,VN:CTD,0.333333,NaT,NaN,NaN,NaN,open,3,0.553264,0.157538,0.158126,0.063390,0.215426,0.405520,NaN


## Step 9 - Generate pyfolio tearsheet


In [32]:
benchmark_rets = None
if config.benchmark:
    from qts.orchestration.flow import _fetch_benchmark_returns

    benchmark_rets = _fetch_benchmark_returns(
        config.benchmark,
        config.start_date,
        config.end_date,
        data_manager,
    )
    if benchmark_rets is not None:
        print(f"Benchmark loaded: {len(benchmark_rets)} days of {config.benchmark}")
    else:
        print(f"Benchmark {config.benchmark} not found in DB - running without")


Benchmark VN:VNINDEX not found in DB - running without


In [33]:
pdf_path = save_tearsheet(
    result=result,
    out_dir=tearsheet_dir(),
    run_id=run_id,
    benchmark_rets=benchmark_rets,
)

if pdf_path:
    print(f"Tearsheet PDF -> {pdf_path}  ({pdf_path.stat().st_size / 1024:.1f} KB)")
else:
    print("Tearsheet skipped (pyfolio not installed or generation failed)")


Start date,2018-01-02
End date,2026-05-29
Total months,99
,Backtest
Annual return,0.997%
Cumulative returns,8.597%
Annual volatility,6.052%
Sharpe ratio,0.19
Calmar ratio,0.08
Stability,0.45
Max drawdown,-13.065%


Worst drawdown periods,Net drawdown in %,Peak date,Valley date,Recovery date,Duration
0,13.06,2023-01-31,2023-02-13,2026-05-19,861
1,2.11,2026-05-19,2026-05-28,NaT,NaN
2,1.96,2023-01-19,2023-01-30,2023-01-31,9
3,1.44,2023-01-06,2023-01-10,2023-01-17,8
4,NaN,NaT,NaT,NaT,NaN


Stress Events,mean,min,max
New Normal,0.00%,0.00%,0.00%
Covid,0.01%,-5.17%,4.79%


Tearsheet PDF -> /Users/s2997726/.qts/exports/backtest/tearsheets/xgb_topk_vectorbt_20260620T233517Z_tearsheet.pdf  (65.2 KB)


In [34]:
# if pdf_path and pdf_path.exists():
#     import shutil
#     import subprocess

#     if shutil.which("pdftoppm"):
#         png_out = pdf_path.with_suffix("")
#         subprocess.run(
#             ["pdftoppm", "-r", "150", "-png", "-l", "1", str(pdf_path), str(png_out)],
#             check=False,
#         )
#         candidates = list(pdf_path.parent.glob(f"{run_id}_tearsheet-*.png"))
#         if candidates:
#             from IPython.display import Image, display

#             display(Image(str(candidates[0])))
#         else:
#             print(f"Open the PDF manually: {pdf_path}")
#     else:
#         print(f"Install poppler for inline display. PDF at: {pdf_path}")


## Summary

| Output | Path |
|--------|------|
| Trade log CSV | `~/.qts/exports/backtest/{run_id}_trade_log.csv` |
| Snapshots CSV | `~/.qts/exports/backtest/{run_id}_snapshots.csv` |
| Tearsheet PDF | `~/.qts/exports/backtest/tearsheets/{run_id}_tearsheet.pdf` |
